# Package 설치

In [1]:
%pip install transformers tokenizers datasets accelerate sentencepiece pillow  timm -qU

Note: you may need to restart the kernel to use updated packages.


# Hugging Face Pipeline을 이용한 모델 활용

- Pipeline은 Transformers 라이브러리의 가장 기본적인 객체로, **전처리 - 추론 -> 후처리** 로 이어지는 일련의 과정을 자동화하여 손쉽게 모델을 사용할 수 있게 해준다.
- Task에 따라 다양한 Pipeline 클래스를 제공하며 `pipeline` 함수를 이용해 쉽게 생성할 수 있다.
- **task만 지정**해서 기본 제공 모델과 토크나이저를 사용하거나 **직접 모델과 토크나이저를 지정**해 생성할 수 있다.
- https://huggingface.co/docs/transformers/pipeline_tutorial

![huggingface_pipeline.png](figures/huggingface_pipeline.png)

## 지원하는 주요 태스크
- https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.pipeline.task
### 자연어 처리 태스크
- **text-classification**: 텍스트 분류
- **text-generation**: 텍스트 생성
- **translation**: 번역
- **summarization**: 요약
- **question-answering**: 질의응답
- **fill-mask**: 마스크 토큰 채우기
- **token-classification**: 개체명 인식, Pos tagging 같이 개별 토큰에 대한 분류
- **feature-extraction**: 특징 추출(context vector)

### 영상 처리 태스크
- **image-classification**: 이미지 분류
- **object-detection**
  -  객체 검출 (Object Detection)
  -  이미지 안에서 객체들의 위치와 class를 찾아내는 작업
- **image-segmentation**
  -  이미지 세분화 (Image Segmentation)
  -  이미지를 픽셀 단위로 분할하여 각 픽셀이 어떤 객체에 속하는지 분류하는 작업

## 모델 검색
![huggingface_model_search.png](figures/huggingface_model_search.png)



## pipeline 함수
- 주요파라미터
  - **task:** 수행하려는 작업의 유형을 문자열로 지정한다.
  - **model:**
    - 사용할 사전 학습된 모델의 이름 또는 경로를 지정한다. 
    - 모델이름(ID)은 `[모델소유자이름]/[모델이름]` 형식이다. Hugging Face에서 제공하는 모델의 경우는 `모델소유자이름`이 생략되어 있다. (ex: "google/gemma-2-2b", "gpt2")
    - 모델을 명시적으로 지정하지 않으면, **task에 맞는 기본 모델이 로드**된다.
  - **tokenizer:** 자연어 task에서 사용할 토크나이저를 지정한다. 생략하면 모델과 같이 제공되는(model과 이름이 같은 토크나이저) 토크나이저를 사용한다.
  - **framework:** 사용할 딥러닝 프레임워크를 지정한다. 'pt'는 PyTorch(Default), 'tf'는 TensorFlow를 지정한다.
  - **device:** Pipeline 모델을 실행할 디바이스를 지정한다. 문자열로 `"cpu", "cuda:1", "mps"`, 또는 GPU 번호를 정수로 지정한다. 
  - **revision:** 모델의 특정 버전을 지정할 때 사용한다.
  - **trust_remote_code:** hub 모델을 직접 다운 받는 것이 아니라 모델을 다운 받는 **코드**를 다운 받아 local에서 실행하는 경우 코드를 실행할 수있게 할 지 여부. (bool)
  - **use_fast:** 
    - 빠른 토크나이저를 사용할지 여부를 지정합니다. 기본값은 True입니다.
    - 빠른 토크나이저는 `Rust` 언어로 구현되어 속도가 빠르다. 단 모든 모델에 대해 지원하지 않는다. 지원하지 않을 경우 `use_fast=True`로 설정해도 일반 토크나이저가 사용된다.

## Task 별 pipeline 실습

### 텍스트 분류

In [1]:
from transformers import pipeline

c:\Users\Playdata\miniconda3\envs\ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 모델과 토크나이저를 로딩해서 Pipeline을 생성
# 모델, 토크나이저를 생략 -> task에 맞는 기본 모델과 토크나이저를 사용.
pipe = pipeline(task="text-classification", framework="pt")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Device set to use cpu


In [4]:
result = pipe("I am very happy.")
result = pipe("I am very unhappy.") # raw -> ||토큰화 -> 추론 -> 후처리 ->|| 최종예측결과

In [5]:
result

[{'label': 'NEGATIVE', 'score': 0.9997710585594177}]

In [6]:
data = [ 
    "The project was completed successfully.", 
    "She always brings positive energy to the team.", 
    "I am confident that we will achieve our goals.",
    "The results were not as expected.", 
    "He struggled to meet the deadline.", 
    "The client was dissatisfied with the final product." 
]
result_list = pipe(data)

In [7]:
result_list

[{'label': 'POSITIVE', 'score': 0.9998227953910828},
 {'label': 'POSITIVE', 'score': 0.9998812675476074},
 {'label': 'POSITIVE', 'score': 0.9998470544815063},
 {'label': 'NEGATIVE', 'score': 0.9978100657463074},
 {'label': 'NEGATIVE', 'score': 0.99960857629776},
 {'label': 'NEGATIVE', 'score': 0.9996129870414734}]

In [8]:
# 특정 모델을 지정해서 사용.
model="distilbert-base-uncased-finetuned-sst-2-english" 
# huggingface에 등록된 모델 ID
## 모델 ID 형식 - 모델소유자ID/모델ID , 모델소유자ID가 생략된 경우: huggingface 자체 모델.
pipe = pipeline(task="text-classification", 
                model=model,  # 사용할 모델을 지정. hf의 모델 ID, 로컬에 저장된 모델저장파일경로.
                tokenizer=model # 사용할 tokenizer를 지정. hf의 토크나이저 ID, 로컬에 저장된 tokenizer 파일경로.
                                # 토크나이저의 ID가 model id와 같을 경우 생략.
                )

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Device set to use cpu


In [9]:
pipe(data)

[{'label': 'POSITIVE', 'score': 0.9998227953910828},
 {'label': 'POSITIVE', 'score': 0.9998812675476074},
 {'label': 'POSITIVE', 'score': 0.9998470544815063},
 {'label': 'NEGATIVE', 'score': 0.9978100657463074},
 {'label': 'NEGATIVE', 'score': 0.99960857629776},
 {'label': 'NEGATIVE', 'score': 0.9996129870414734}]

In [10]:
kor_texts = [
    "이 영화 정말 재미있어요!",
    "서비스가 별로였어요.",
    "제품 품질이 우수합니다.",
    "따듯하고 부드럽고 제품은 너무 좋습니다. 그런데 배송이 너무 늦네요."  # 애매한 것 0.56 정도 나오네.
]

In [11]:
pipe(kor_texts)

[{'label': 'POSITIVE', 'score': 0.9855567812919617},
 {'label': 'POSITIVE', 'score': 0.7425776124000549},
 {'label': 'POSITIVE', 'score': 0.6555716395378113},
 {'label': 'NEGATIVE', 'score': 0.5247918367385864}]

In [12]:
model = 'Copycats/koelectra-base-v3-generalized-sentiment-analysis' 
pipe = pipeline(task="text-classification", model=model)
result_list = pipe(kor_texts)

Device set to use cpu


In [13]:
result_list  # 1: 긍정, 0: 부정

[{'label': '1', 'score': 0.9897311329841614},
 {'label': '0', 'score': 0.9969298243522644},
 {'label': '1', 'score': 0.9640172123908997},
 {'label': '0', 'score': 0.5669127702713013}]

In [14]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis")

Device set to use cpu


In [15]:
pipe("오늘은 날씨가 너무 좋다.")

[{'label': 'Very Positive', 'score': 0.665915846824646}]

In [16]:
kor_texts = [
    "이 영화 정말 재미있어요!",
    "서비스가 별로였어요.",
    "제품 품질이 우수합니다.",
    "따듯하고 부드럽고 제품은 너무 좋습니다. 그런데 배송이 너무 늦네요."  # 애매한 것 0.56 정도 나오네.
]

In [17]:
pipe(kor_texts)

[{'label': 'Very Positive', 'score': 0.5016762614250183},
 {'label': 'Negative', 'score': 0.5245441794395447},
 {'label': 'Very Positive', 'score': 0.6271345615386963},
 {'label': 'Very Positive', 'score': 0.5461609959602356}]

In [18]:
result_list = pipe(data)

In [19]:
for txt, r in zip(data, result_list):
    print(txt, r)

The project was completed successfully. {'label': 'Neutral', 'score': 0.49546512961387634}
She always brings positive energy to the team. {'label': 'Positive', 'score': 0.6366152167320251}
I am confident that we will achieve our goals. {'label': 'Neutral', 'score': 0.45890164375305176}
The results were not as expected. {'label': 'Negative', 'score': 0.6052263975143433}
He struggled to meet the deadline. {'label': 'Negative', 'score': 0.518038272857666}
The client was dissatisfied with the final product. {'label': 'Negative', 'score': 0.6167647838592529}


### 제로샷 분류
- 제로샷(Zero-shot)은 각 개별 작업에 대한 특정 교육 없이 작업을 수행할 수 있는 task다.
- 입력 텍스트와 함께 클래스 레이블을 제공하면 분류 작업을 한다.
- 모델은  `task`에서 `Zero-Shot` 으로 시작하는 task를 선택하여 검색한다.

In [20]:
model = "facebook/bart-large-mnli"

text = ["Python is a programming language.", 
        "I love soccer", 
        "The stock price rose slightly today."]

labels = ["IT", "Sports"]

In [21]:
pipe = pipeline(task="zero-shot-classification", model=model)
result = pipe(text, candidate_labels=labels)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Device set to use cpu


In [22]:
result

[{'sequence': 'Python is a programming language.',
  'labels': ['IT', 'Sports'],
  'scores': [0.5758535265922546, 0.4241464138031006]},
 {'sequence': 'I love soccer',
  'labels': ['Sports', 'IT'],
  'scores': [0.9935312867164612, 0.006468690931797028]},
 {'sequence': 'The stock price rose slightly today.',
  'labels': ['IT', 'Sports'],
  'scores': [0.6849520802497864, 0.3150479197502136]}]

In [23]:
labels = ["business", "programming", "sports", "movie", "education"]
result = pipe(text, candidate_labels=labels)
result

[{'sequence': 'Python is a programming language.',
  'labels': ['programming', 'business', 'movie', 'sports', 'education'],
  'scores': [0.9856367111206055,
   0.005072721280157566,
   0.0034023483749479055,
   0.002961924998089671,
   0.0029262355528771877]},
 {'sequence': 'I love soccer',
  'labels': ['sports', 'programming', 'business', 'movie', 'education'],
  'scores': [0.9952405691146851,
   0.0012840895215049386,
   0.0012676474871113896,
   0.0012649551499634981,
   0.0009427034528926015]},
 {'sequence': 'The stock price rose slightly today.',
  'labels': ['business', 'movie', 'programming', 'sports', 'education'],
  'scores': [0.7462778091430664,
   0.06974831968545914,
   0.06889291107654572,
   0.0645080953836441,
   0.05057287961244583]}]

### 텍스트 생성

In [24]:
pipe = pipeline(task="text-generation")

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d (https://huggingface.co/openai-community/gpt2).
Using a pipeline without specifying a model name and revision in production is not recommended.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Device set to use cpu


In [25]:
start_text = "Today weather"
sent = pipe(start_text)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [26]:
sent

[{'generated_text': 'Today weather, the temperature rises. And it is not just the heat that causes an increase in temperature but also the way that it is measured.\n\nFor example, if the temperature rises by 3 degrees Fahrenheit, then you can expect the water to cool by about 35 degrees Fahrenheit. At 5 degrees Fahrenheit, you can expect the water to cool by about 30 degrees Fahrenheit. That is, the water will cool by about 50 degrees Fahrenheit or so. For the same reason, if the temperature rises by 4 degrees Fahrenheit, the water will cool by about 30 degrees Fahrenheit.\n\nIn all of these ways, it is possible to measure the temperature at different times of the day. For example, in the summer time, the temperature drops and the water will cool by about 30 degrees Fahrenheit. In the winter, the temperature drops and the water will cool by about 30 degrees Fahrenheit.\n\nThe temperature records by the time of day are also not very accurate. In many models, it is necessary to use a tim

In [27]:
print(sent[0]["generated_text"])

Today weather, the temperature rises. And it is not just the heat that causes an increase in temperature but also the way that it is measured.

For example, if the temperature rises by 3 degrees Fahrenheit, then you can expect the water to cool by about 35 degrees Fahrenheit. At 5 degrees Fahrenheit, you can expect the water to cool by about 30 degrees Fahrenheit. That is, the water will cool by about 50 degrees Fahrenheit or so. For the same reason, if the temperature rises by 4 degrees Fahrenheit, the water will cool by about 30 degrees Fahrenheit.

In all of these ways, it is possible to measure the temperature at different times of the day. For example, in the summer time, the temperature drops and the water will cool by about 30 degrees Fahrenheit. In the winter, the temperature drops and the water will cool by about 30 degrees Fahrenheit.

The temperature records by the time of day are also not very accurate. In many models, it is necessary to use a time-series record to look at 

In [28]:
pipe(["I am", "Python is", "LLM is"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[[{'generated_text': 'I am very sorry for the inconvenience caused to you. We do not want to send you any further problems because we are not quite sure how to fix it. However, you are welcome to contact us if you have any additional questions or to let us know what you are experiencing.\n\n\nWe also have some additional questions regarding your system. Please contact us if you see any problems or any other questions.\n\n\nWe do not expect any trouble from you. We hope your understanding will be helpful.'}],
 [{'generated_text': "Python is a fairly simple language. It's a little bit of a pain to learn, but it's a solid system that can be used in most situations.\n\nI've tried to use it with a variety of project and it's a bit of a pain to learn. A lot of time I've spent searching for solutions to problems that I have and I feel like I've spent a lot of time at StackOverflow. So I think I've reached the right balance.\n\nAs a developer, however, you also need to understand the concepts 

In [29]:
pipe("나는 어제 ")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': '나는 어제 갭어스 과각는 어도나 난나이 세무스을 과이 다은 있 있데 하는 합니다. 나가 인선 그면 기상지의 할지 다은 있 있데 하는 합니다. 가로는 오러요 하과 감는 여드하 과가 나는 여드고 이에 지시에 그렇은 그망을 감는 있어�'}]

In [30]:
from transformers import pipeline

model_id = 'Qwen/Qwen3-0.6B'
pipe = pipeline(task="text-generation", model=model_id)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Device set to use cpu


In [31]:
pipe("나는 어제")

[{'generated_text': '나는 어제 금요일 저녁에 300대의 친구를 만났어요. 그 친구는 캐치마크를 사용했다. 그 친구는 캐치마크를 사용했을 때의 행동은 무엇이라고 할 수 있나요?\n\nA. 친구가 친구를 위한 캐치마크를 사용했다. B. 친구가 친구를 위한 캐치마크를 사용했다. C. 친구가 친구를 위한 캐치마크를 사용했다. D. 친구가 친구를 위한 캐치마크를 사용했다.\n\nAnswer:\nThe correct answer is A.\n\nBut wait, I need to be careful here. The question is asking about the action that the friend took when using the charm. The options all say "used the charm," which is the same as the action. The answer should be A. So even though the options are all the same, the correct answer is A. I think that\'s right.\n\nBut maybe the answer is different. Let me check again. The question is asking what the friend\'s action was when using the charm. The options are'}]

In [32]:
msg = [
    {"role":"user", "content":"LLM에 대해서 설명해줘."}
]
result = pipe(msg, max_new_tokens=1000)
#  max_new_tokens : 응답 토큰수 설정.

In [33]:
result[0]["generated_text"][0]  # user 입력(query)

{'role': 'user', 'content': 'LLM에 대해서 설명해줘.'}

In [34]:
result[0]["generated_text"][1] # AI 답변

{'role': 'assistant',
 'content': '<think>\nOkay, the user asked for an explanation of LLM. First, I need to recall what I know about LLMs. I remember they\'re large language models, which are trained on vast amounts of text data. They can understand and generate text, but there are some limitations.\n\nI should start by defining LLM. Then explain that they are trained on huge datasets, which makes them very powerful. Mention their ability to understand and generate text, but also talk about their limitations, like not knowing all possible answers or being biased if the data is biased. \n\nIt\'s important to highlight their strengths and the challenges they face. Also, maybe mention how they are used in various fields like language processing, customer service, and creative writing. \n\nWait, should I include examples? Like, how they can answer questions or write stories. But maybe keep it general. Make sure the explanation is clear and covers both the positives and the negatives. Avoi

In [35]:
print(result[0]["generated_text"][1]['content'])

<think>
Okay, the user asked for an explanation of LLM. First, I need to recall what I know about LLMs. I remember they're large language models, which are trained on vast amounts of text data. They can understand and generate text, but there are some limitations.

I should start by defining LLM. Then explain that they are trained on huge datasets, which makes them very powerful. Mention their ability to understand and generate text, but also talk about their limitations, like not knowing all possible answers or being biased if the data is biased. 

It's important to highlight their strengths and the challenges they face. Also, maybe mention how they are used in various fields like language processing, customer service, and creative writing. 

Wait, should I include examples? Like, how they can answer questions or write stories. But maybe keep it general. Make sure the explanation is clear and covers both the positives and the negatives. Avoid technical jargon but explain the concepts 

### 마스크 채우기

In [36]:
text = "I'm going to <mask> because <mask> am hurt."
model="distilroberta-base"

pipe = pipeline(task="fill-mask", model=model)
result = pipe(text, top_k=2) # <mask>에 들어갈 확률이 가장높은 단어 2개(top_k=2)를 찾기.

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of the model checkpoint at distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [37]:
result

[[{'score': 0.26520147919654846,
   'token': 8930,
   'token_str': ' cry',
   'sequence': "<s>I'm going to cry because<mask> am hurt.</s>"},
  {'score': 0.06089096516370773,
   'token': 3581,
   'token_str': ' sleep',
   'sequence': "<s>I'm going to sleep because<mask> am hurt.</s>"}],
 [{'score': 0.9930052161216736,
   'token': 38,
   'token_str': ' I',
   'sequence': "<s>I'm going to<mask> because I am hurt.</s>"},
  {'score': 0.006336296442896128,
   'token': 939,
   'token_str': ' i',
   'sequence': "<s>I'm going to<mask> because i am hurt.</s>"}]]

In [38]:
text = "오늘 밤은 전국이 흐린 가운데 대부분 지역에 [MASK]가 내리겠고, 기온이 내려가면서 점차 [MASK]이 오는 곳이 많겠습니다"
# pipe(text, top_k=2) # roberta모델은 한글을 학습하지 않은 모델.

model='beomi/kcbert-base'
pipe = pipeline(task='fill-mask', model=model)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of the model checkpoint at beomi/kcbert-base were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [39]:
pipe(text)

[[{'score': 0.6340416669845581,
   'token': 4072,
   'token_str': '##서',
   'sequence': '[CLS] 오늘 밤은 전국이 흐린 가운데 대부분 지역에서 가 내리겠고, 기온이 내려가면서 점차 [MASK] 이 오는 곳이 많겠습니다 [SEP]'},
  {'score': 0.11311744153499603,
   'token': 28206,
   'token_str': '비가',
   'sequence': '[CLS] 오늘 밤은 전국이 흐린 가운데 대부분 지역에 비가 가 내리겠고, 기온이 내려가면서 점차 [MASK] 이 오는 곳이 많겠습니다 [SEP]'},
  {'score': 0.03714243322610855,
   'token': 12,
   'token_str': ')',
   'sequence': '[CLS] 오늘 밤은 전국이 흐린 가운데 대부분 지역에 ) 가 내리겠고, 기온이 내려가면서 점차 [MASK] 이 오는 곳이 많겠습니다 [SEP]'},
  {'score': 0.035172488540410995,
   'token': 1664,
   'token_str': '비',
   'sequence': '[CLS] 오늘 밤은 전국이 흐린 가운데 대부분 지역에 비 가 내리겠고, 기온이 내려가면서 점차 [MASK] 이 오는 곳이 많겠습니다 [SEP]'},
  {'score': 0.0196221936494112,
   'token': 9666,
   'token_str': '##서는',
   'sequence': '[CLS] 오늘 밤은 전국이 흐린 가운데 대부분 지역에서는 가 내리겠고, 기온이 내려가면서 점차 [MASK] 이 오는 곳이 많겠습니다 [SEP]'}],
 [{'score': 0.10058414191007614,
   'token': 10108,
   'token_str': '바람',
   'sequence': '[CLS] 오늘 밤은 전국이 흐린 가운데 대부분 지역에 [MASK] 가 내리겠고,

### Token별 분류
- task: token-classification 
  - 개체명인식(ner), 품사부착(pos tagging)을 수행하는 task 
  - 개체명 인식은 문장에서 특정한 개체명(예: 사람 이름, 지명, 조직명 등)을 식별하는 task이다. 

In [40]:
text = "My name is Sylvain and I work at Hugging Face in Brooklyn."
model = "dbmdz/bert-large-cased-finetuned-conll03-english"

pipe = pipeline(task='token-classification', model=model)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [41]:
result = pipe(text)
result

[{'entity': 'I-PER',
  'score': 0.99938285,
  'index': 4,
  'word': 'S',
  'start': 11,
  'end': 12},
 {'entity': 'I-PER',
  'score': 0.99815494,
  'index': 5,
  'word': '##yl',
  'start': 12,
  'end': 14},
 {'entity': 'I-PER',
  'score': 0.99590707,
  'index': 6,
  'word': '##va',
  'start': 14,
  'end': 16},
 {'entity': 'I-PER',
  'score': 0.99923277,
  'index': 7,
  'word': '##in',
  'start': 16,
  'end': 18},
 {'entity': 'I-ORG',
  'score': 0.9738931,
  'index': 12,
  'word': 'Hu',
  'start': 33,
  'end': 35},
 {'entity': 'I-ORG',
  'score': 0.976115,
  'index': 13,
  'word': '##gging',
  'start': 35,
  'end': 40},
 {'entity': 'I-ORG',
  'score': 0.9887976,
  'index': 14,
  'word': 'Face',
  'start': 41,
  'end': 45},
 {'entity': 'I-LOC',
  'score': 0.9932106,
  'index': 16,
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

### 질의 응답
- 문서와 질문을 주면 문서에서 답을 찾아 응답한다.

In [42]:
model = "distilbert-base-cased-distilled-squad"
pipe = pipeline(task="question-answering", model=model)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Device set to use cpu


In [43]:
question="Where do I work?"
question="Where is Hugging Face?"
context="My name is Sylvain and I work at Hugging Face in Brooklyn"

result = pipe(question=question,  # 질문
              context=context)    # 답을 찾을 문서

In [44]:
result

{'score': 0.9893267154693604, 'start': 49, 'end': 57, 'answer': 'Brooklyn'}

In [45]:
context = """우리나라 2대 수출 품목인 자동차가 도널드 트럼프 미국 행정부의 관세 여파로 지난달 큰 폭의 수출 감소율을 보이면서 우려가 커지고 있다. 현대차, 기아의 미국 수출 비중이 최대 85%에 이르는 상황에서 자동차 관세 장기화 시 피해는 걷잡을 수 없이 불어날 것이라는 암울한 전망이 나온다.
1일 산업통상자원부가 발표한 5월 수출입 동향에 따르면 지난달 자동차 수출은 작년 동기 대비 4.4% 감소한 62억달러로 집계됐다. 최대 자동차 시장인 미국으로의 수출은 18억4000만달러로 무려 32.0% 급감했다.
4월 미국의 수입산 자동차 25% 관세 부과에 이어 5월부터 일부 자동차 부품에도 25%의 관세가 적용된 결과다. 관세 장기화 시 피해는 더 커질 것이라는 우려가 현실화한 셈이다.
국내 완성차 1·2위 업체인 현대차·기아는 현지 생산 비중을 확대하는 동시에 가격 인상을 검토하고 있다. 관세 여파를 흡수하기 위해서다. 가격 인상이 현실화할 경우 미국 현지 판매는 줄어들 수밖에 없어 수출에는 더 악영향을 미칠 것으로 보인다.
"""

q1 = "현대차 기아의 미국 수출비중은?"
q2 = "자동차 수출이 얼마나 급감했나?"
q3 = "대미 수출 감소에 국내 자동차 업체들의 대응방법은?"

In [46]:
model = "ainize/klue-bert-base-mrc"
pipe = pipeline(task='question-answering', model=model)

Device set to use cpu


In [47]:
result = pipe(question=[q1, q2, q3], context=context)

In [48]:
result

[{'score': 0.6396173238754272, 'start': 99, 'end': 102, 'answer': '85%'},
 {'score': 0.632358193397522, 'start': 271, 'end': 276, 'answer': '32.0%'},
 {'score': 0.013693439774215221, 'start': 427, 'end': 433, 'answer': '가격 인상을'}]

### 문서 요약

In [49]:
model = "eenzeenee/t5-base-korean-summarization"

### 번역

In [50]:
model = "Helsinki-NLP/opus-mt-fr-en"
text = "Ce cours est produit par Hugging Face."

In [51]:
model = "Helsinki-NLP/opus-mt-ko-en"

In [58]:
res

NameError: name 'res' is not defined

### 이미지를 설명하는 텍스트 생성

In [61]:
url1 = "https://huggingface.co/datasets/Narsil/image_dummy/resolve/main/parrots.png"
url2 = "https://th.bing.com/th?id=ORMS.c526884bbea37c0bb9501f4f83b601e4&pid=Wdp&w=268&h=140&qlt=90&c=1&rs=1&dpr=1&p=0"
url3 = "http://images.cocodataset.org/val2017/000000039769.jpg"

In [59]:
model = "ydshieh/vit-gpt2-coco-en"

### 이미지 분류

In [54]:
url = "https://pds.joongang.co.kr/news/component/htmlphoto_mmdata/202306/25/488f9638-800c-4bac-ad65-82877fbff79b.jpg"

In [ ]:
model = "google/vit-base-patch16-224"


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Device set to use cpu


AttributeError: 'ImageToTextPipeline' object has no attribute 'assistant_model'

### Object Detection

In [56]:
image_path = r"data/image1.jpg"
image_path = r"data/image2.jpg"
image_path = r"data/image3.jpg"

model='facebook/detr-resnet-50'